In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 获取当前文件夹下所有名为 data* 的Excel文件
files = [f for f in os.listdir() if f.startswith('data') and (f.endswith('.xlsx') or f.endswith('.xls'))]

# 初始化变量来存储最小日期
min_date = None

# 获取所有文件中最小的日期
for file in files:
    try:
        # 读取Excel文件
        df = pd.read_excel(file)
        
        if df.shape[1] >= 2:  # 确保有足够的列
            # 获取倒数第二列的Date数据
            date_col = df.iloc[:, -2].astype(str)
            
            # 将Date列转换为datetime对象
            date_col = pd.to_datetime(date_col, format='%Y%m%d')

            # 获取最小日期
            file_min_date = date_col.min()
            
            # 更新最小日期
            if min_date is None or file_min_date < min_date:
                min_date = file_min_date

    except Exception as e:
        print(f"读取文件 {file} 时出错: {e}")

# 检查是否找到了有效的最小日期
if min_date is None:
    print("无法找到有效的最小日期，程序退出。")
    exit()

# 添加进度条处理每个文件
for file in tqdm(files, desc="Processing files"):
    try:
        # 读取Excel文件
        df = pd.read_excel(file)
        
        # 确保文件中至少有两列，以确保存在倒数第二列
        if df.shape[1] >= 2:
            # 获取倒数第二列的Date数据
            date_col = df.iloc[:, -2].astype(str)
            
            # 将Date列转换为datetime对象
            date_col = pd.to_datetime(date_col, format='%Y%m%d')
            
            # 计算每一行数据的累计天数
            cumulative_days = (date_col - min_date).dt.days
            
            # 将计算出的累计天数放到最后一列，并命名为 Cumulative_days
            df['Cumulative_days'] = cumulative_days
            
            # 保存修改后的数据到原文件
            df.to_excel(file, index=False)
            print(f"{file} 已处理完毕。")
        else:
            print(f"{file} 没有足够的列，跳过该文件。")
    except Exception as e:
        print(f"处理文件 {file} 时出错: {e}")

print("所有文件处理完毕。")
